# 03 — Feature provenance, preprocessing, and split design

**Purpose:** create a reproducible, leakage-aware preprocessing and evaluation plan.

This notebook intentionally does not fit a model. First resolve the label-generation process and complete the feature decision register in Notebook 02.

## 1. Load data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Put the CSV beside this notebook, or change DATA_PATH to its full path.
DATA_PATH = Path("metaverse_transactions_dataset.csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the original CSV beside this notebook "
        f"or edit DATA_PATH. CSV files in this folder: {[p.name for p in candidates]}"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Shape: (78600, 14)


,timestamp,hour_of_day,sending_address,receiving_address,amount,transaction_type,location_region,ip_prefix,login_frequency,session_duration,purchase_pattern,age_group,risk_score,anomaly
0,2022-04-11 12:47:27,12,0x9d32d0bf2c00f41ce7ca01b66e174cc4dcb0c1da,0x39f82e1c09bc6d7baccc1e79e5621ff812f50572,796.949206,transfer,Europe,192.000,3,48,focused,established,18.75,low_risk
1,2022-06-14 19:12:46,19,0xd6e251c23cbf52dbd472f079147873e655d8096f,0x51e8fbe24f124e0e30a614e14401b9bbfed5384c,0.010000,purchase,South America,172.000,5,61,focused,established,25.00,low_risk
2,2022-01-18 16:26:59,16,0x2e0925b922fed01f6a85d213ae2718f54b8ca305,0x52c7911879f783d590af45bda0c0ef2b8536706f,778.197390,purchase,Asia,192.168,3,74,focused,established,31.25,low_risk
3,2022-06-15 09:20:04,9,0x93efefc25fcaf31d7695f28018d7a11ece55457f,0x8ac3b7bd531b3a833032f07d4e47c7af6ea7bace,300.838358,transfer,South America,172.000,8,111,high_value,veteran,36.75,low_risk
4,2022-02-18 14:35:30,14,0xad3b8de45d63f5cce28aef9a82cf30c397c6ceb9,0x6fdc047c2391615b3facd79b4588c7e9106e49f2,775.569344,sale,Africa,172.160,6,100,high_value,veteran,62.50,moderate_risk


## 2. Declare target and candidate feature decisions

In [2]:
TARGET = "anomaly"

# Conservative provisional exclusions. Review these against source-code provenance.
# Add columns only after documenting that they are available before prediction
# and are not used to construct the target.
PROVISIONAL_EXCLUDE = {
    TARGET,
    "risk_score",       # previously observed to perfectly separate target classes
    "transaction_type", # target-aligned pattern requires provenance review
    "purchase_pattern", # provenance not yet established
    "age_group",        # provenance not yet established
}

candidate_features = [c for c in df.columns if c not in PROVISIONAL_EXCLUDE]
print("Provisional exclusions:", sorted(PROVISIONAL_EXCLUDE))
print("Candidate features:", candidate_features)

Provisional exclusions: ['age_group', 'anomaly', 'purchase_pattern', 'risk_score', 'transaction_type']
Candidate features: ['timestamp', 'hour_of_day', 'sending_address', 'receiving_address', 'amount', 'location_region', 'ip_prefix', 'login_frequency', 'session_duration']


## 3. Review identifiers and post-outcome fields

In [3]:
possible_identifiers = [c for c in df.columns if any(
    token in c.lower() for token in ["id", "hash", "address", "wallet", "account", "user"]
)]
print("Potential identifiers / grouping fields:", possible_identifiers)
print("Review whether records from the same wallet, user, or transaction family can cross splits.")

Potential identifiers / grouping fields: ['sending_address', 'receiving_address']
Review whether records from the same wallet, user, or transaction family can cross splits.


## 4. Timestamp-based split recommendation

In [4]:
if "timestamp" in df.columns:
    temp = df.copy()
    temp["_parsed_timestamp"] = pd.to_datetime(temp["timestamp"], errors="coerce", utc=True)
    print("Rows with invalid timestamps:", int(temp["_parsed_timestamp"].isna().sum()))
    print("Time range:", temp["_parsed_timestamp"].min(), "to", temp["_parsed_timestamp"].max())
    print("Chronological split is preferable when deployment predicts future transactions.")
else:
    print("No timestamp column; justify an alternative split strategy.")

Rows with invalid timestamps: 0
Time range: 2022-01-01 00:00:45+00:00 to 2022-12-31 23:47:40+00:00
Chronological split is preferable when deployment predicts future transactions.


## 5. Repeated-entity leakage checks

In [5]:
for col in possible_identifiers:
    if col in df.columns:
        counts = df[col].value_counts(dropna=False)
        print(f"{col}: {len(counts)} unique values; {int((counts > 1).sum())} values repeated")

sending_address: 1161 unique values; 1161 values repeated
receiving_address: 1166 unique values; 1166 values repeated


## 6. Preprocessing blueprint (do not fit until feature decisions are approved)

Use a scikit-learn `Pipeline` and `ColumnTransformer` so imputers, encoders, and scalers are fitted on training data only. Never preprocess the full dataset before splitting.

## 7. Required provenance register

For every candidate feature, record:
- definition and unit;
- source system/table;
- timestamp when it becomes available;
- whether it is computed from the target, a risk score, or a post-transaction outcome;
- whether it is an identifier or proxy for identity;
- keep/exclude decision and rationale.